# Email Spam Detection - KNN vs SVM
Binary classification of emails into two states:
* **Normal state (0)** - Not Spam
* **Abnormal state (1)** - Spam

Models: **K-Nearest Neighbors (KNN)** and **Support Vector Machine (SVM)**, followed by a performance analysis (accuracy, precision, recall, F1, confusion matrix, ROC-AUC, cross-validation).

> Dataset: `emails.csv` (Kaggle). Each row is an email, each column is the count of a frequent word, and `Prediction` is the label. Keep `emails.csv` in the same folder as this notebook.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn import metrics

sns.set_style("whitegrid")
RANDOM_STATE = 42

## 2. Load and explore the dataset

In [ ]:
df = pd.read_csv("emails.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
print("Number of feature columns:", df.shape[1] - 2, "(excluding 'Email No.' and 'Prediction')")
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df[["Prediction"]].describe().T

### Class distribution (Not Spam vs Spam)

In [ ]:
counts = df["Prediction"].value_counts().sort_index()
labels = {0: "Not Spam (Normal)", 1: "Spam (Abnormal)"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar([labels[i] for i in counts.index], counts.values, color=["#2ca02c", "#d62728"])
axes[0].set_title("Number of emails per class")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom")
axes[1].pie(counts.values, labels=[labels[i] for i in counts.index], autopct="%1.1f%%",
            colors=["#2ca02c", "#d62728"], startangle=90)
axes[1].set_title("Class share")
plt.tight_layout()
plt.show()

The classes are imbalanced (far more *Not Spam* than *Spam*), so accuracy alone can be misleading. We therefore also look at **precision, recall, F1 and ROC-AUC** for the spam class.

## 3. Pre-processing

In [ ]:
# Handle missing values / duplicates, drop the identifier column
df = df.dropna().drop_duplicates()

X = df.drop(columns=["Email No.", "Prediction"])
y = df["Prediction"]
print("Features:", X.shape, " Target:", y.shape)

In [ ]:
# Stratified split keeps the spam ratio equal in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Spam share  -> train: %.3f, test: %.3f" % (y_train.mean(), y_test.mean()))

**Feature scaling:** KNN (distance based) and SVM (margin based) are both sensitive to feature scale. The `StandardScaler` is placed **inside a pipeline**, so it is fitted on the training data only and there is no data leakage from the test set.

## 4. K-Nearest Neighbors (KNN)

### 4.1 Choose the best K

In [ ]:
k_values = range(1, 26, 2)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = []
for k in k_values:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_scores.append(cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean())

best_k = list(k_values)[int(np.argmax(cv_scores))]

plt.figure(figsize=(8, 4))
plt.plot(list(k_values), cv_scores, marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"best K = {best_k}")
plt.xlabel("K (number of neighbours)"); plt.ylabel("5-fold CV F1-score (spam)")
plt.title("KNN: choosing K"); plt.legend()
plt.show()
print("Best K:", best_k)

### 4.2 Train and predict

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)
y_prob_knn = knn.predict_proba(X_test)[:, 1]
print("KNN accuracy:", round(metrics.accuracy_score(y_test, y_pred_knn), 4))

## 5. Support Vector Machine (SVM)

### 5.1 Compare kernels

In [ ]:
kernel_scores = {}
for kernel in ["linear", "rbf", "poly", "sigmoid"]:
    pipe = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1))
    kernel_scores[kernel] = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()

pd.Series(kernel_scores, name="5-fold CV F1 (spam)").round(4).to_frame()

In [ ]:
best_kernel = max(kernel_scores, key=kernel_scores.get)
print("Best kernel:", best_kernel)

# Effect of the cost parameter C for the best kernel
C_values = [0.01, 0.1, 1, 10, 100]
c_scores = []
for C in C_values:
    pipe = make_pipeline(StandardScaler(), SVC(kernel=best_kernel, C=C))
    c_scores.append(cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean())

plt.figure(figsize=(8, 4))
plt.semilogx(C_values, c_scores, marker="o")
plt.xlabel("C (cost)"); plt.ylabel("5-fold CV F1-score (spam)")
plt.title(f"SVM ({best_kernel} kernel): choosing C")
plt.show()

best_C = C_values[int(np.argmax(c_scores))]
print("Best C:", best_C)

### 5.2 Train and predict

In [ ]:
svm = make_pipeline(StandardScaler(), SVC(kernel=best_kernel, C=best_C))
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)
y_score_svm = svm.decision_function(X_test)
print("SVM accuracy:", round(metrics.accuracy_score(y_test, y_pred_svm), 4))

## 6. Performance analysis

### 6.1 Confusion matrices

In [ ]:
class_names = ["Not Spam", "Spam"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name, pred in zip(axes, ["KNN", "SVM"], [y_pred_knn, y_pred_svm]):
    cm = metrics.confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set(title=f"{name} - confusion matrix", xlabel="Predicted", ylabel="Actual")
plt.tight_layout()
plt.show()

for name, pred in [("KNN", y_pred_knn), ("SVM", y_pred_svm)]:
    tn, fp, fn, tp = metrics.confusion_matrix(y_test, pred).ravel()
    print(f"{name}: TN={tn}  FP={fp} (good mail marked spam)  FN={fn} (spam missed)  TP={tp}")

### 6.2 Classification reports

In [ ]:
print("=== KNN ===")
print(metrics.classification_report(y_test, y_pred_knn, target_names=class_names, digits=4))
print("=== SVM ===")
print(metrics.classification_report(y_test, y_pred_svm, target_names=class_names, digits=4))

### 6.3 Comparison table

In [ ]:
def summarise(name, y_true, y_pred, y_score):
    return {
        "Model": name,
        "Accuracy": metrics.accuracy_score(y_true, y_pred),
        "Precision (spam)": metrics.precision_score(y_true, y_pred),
        "Recall (spam)": metrics.recall_score(y_true, y_pred),
        "F1 (spam)": metrics.f1_score(y_true, y_pred),
        "ROC-AUC": metrics.roc_auc_score(y_true, y_score),
    }

results = pd.DataFrame([
    summarise(f"KNN (K={best_k})", y_test, y_pred_knn, y_prob_knn),
    summarise(f"SVM ({best_kernel}, C={best_C})", y_test, y_pred_svm, y_score_svm),
]).set_index("Model").round(4)
results

In [ ]:
results.T.plot(kind="bar", figsize=(10, 4.5), rot=0, ylim=(0, 1.05))
plt.title("KNN vs SVM - test-set performance")
plt.legend(loc="lower right")
plt.show()

### 6.4 ROC curves

In [ ]:
plt.figure(figsize=(6.5, 5.5))
for name, score in [("KNN", y_prob_knn), ("SVM", y_score_svm)]:
    fpr, tpr, _ = metrics.roc_curve(y_test, score)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {metrics.roc_auc_score(y_test, score):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curve"); plt.legend()
plt.show()

### 6.5 Cross-validation (stability check on the whole dataset)

In [ ]:
cv_results = {}
for name, model in [("KNN", knn), ("SVM", svm)]:
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    cv_results[name] = scores

cv_df = pd.DataFrame(cv_results)
cv_df.loc["Mean"] = cv_df.mean()
cv_df.loc["Std"] = cv_df.iloc[:-1].std()
cv_df.round(4)

## 7. Predict a new email

In [ ]:
# Take a few unseen emails from the test set and show what each model says
sample = X_test.iloc[:8]
out = pd.DataFrame({
    "Actual": y_test.iloc[:8].map({0: "Not Spam", 1: "Spam"}).values,
    "KNN": pd.Series(knn.predict(sample)).map({0: "Not Spam", 1: "Spam"}).values,
    "SVM": pd.Series(svm.predict(sample)).map({0: "Not Spam", 1: "Spam"}).values,
})
out

## 8. Conclusion

* The dataset has two states - **Not Spam (normal)** and **Spam (abnormal)** - and is imbalanced, so precision, recall, F1 and ROC-AUC were used besides accuracy.
* **Scaling** was done inside a pipeline, and K (KNN) and kernel/C (SVM) were chosen using cross-validation on the training set only.
* **Which model is better?** Compare the table in Section 6.3:
  * **Recall (spam)** - how many real spam emails were caught (`FN` = spam that slipped into the inbox).
  * **Precision (spam)** - how many emails flagged as spam really were spam (`FP` = good mail wrongly blocked; usually the costlier mistake).
  * SVM is typically the stronger and more stable model on this kind of high-dimensional word-count data, while KNN is simple but slower at prediction time and more sensitive to the many irrelevant features.
* **Possible improvements:** TF-IDF features, feature selection/PCA, `GridSearchCV` over more parameters, class weights (`class_weight="balanced"`) and other models (Naive Bayes, Logistic Regression).